# CUDA Tutorial - Automated GPU Execution and Verification
**Course:** High-Performance Computing (HPC)  
**Target Hardware:** NVIDIA GPU (Google Colab Tesla T4 / A100)  
**Repository:** [https://github.com/yaswanthkumar112006-maker/hpc.git](https://github.com/yaswanthkumar112006-maker/hpc.git)

---
### How to Run in Google Colab (2 Simple Steps):
1. **Enable GPU:** In the top menu, click **Runtime** > **Change runtime type**, select **T4 GPU**, and click **Save**.
2. **Run All:** Click **Runtime** > **Run all** (or press `Ctrl+F9`).

This notebook automatically sets up directories, verifies your GPU, prepares all 5 CUDA source files, compiles them with `nvcc -O3`, executes them, verifies correctness, and downloads genuine output logs.

## Step 1: Verify NVIDIA GPU & CUDA Compiler

In [ ]:
import subprocess
import sys

# Verify GPU is detected
res_gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if res_gpu.returncode != 0:
    print('❌ ERROR: No NVIDIA GPU detected! Please enable GPU in Runtime > Change runtime type > T4 GPU.')
    raise SystemExit('Please enable GPU runtime to proceed.')
else:
    print('✅ NVIDIA GPU detected successfully:\n')
    print(res_gpu.stdout)

# Verify NVCC compiler
!nvcc --version

## Step 2: Prepare Workspace & Ensure All CUDA Source Files Exist
This step sets up `/content/Tutorial-CUDA` with `src/`, `bin/`, and `results/` folders, and writes all 5 source files automatically so there are no missing path errors.

In [ ]:
import os
import json
import subprocess

# Set working directory to /content/Tutorial-CUDA
base_dir = "/content/Tutorial-CUDA"
os.makedirs(base_dir, exist_ok=True)
os.chdir(base_dir)

# Create standard subdirectories
os.makedirs("src", exist_ok=True)
os.makedirs("bin", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("screenshots", exist_ok=True)

# Write all 5 latest source files directly into src/
SOURCES = {"src/01_device_query.cu": "/**\n * Tutorial-CUDA: Task 1 - Device Properties Query\n * \n * Queries and displays comprehensive hardware and architectural properties\n * for all available CUDA-capable devices on the system using cudaGetDeviceProperties.\n *\n * Properties queried:\n * a. Device name\n * b. Total global memory\n * c. Shared memory per block\n * d. Number of multiprocessors (SMs)\n * e. Maximum threads per block\n * f. Maximum grid size (x, y, z)\n * g. Compute capability (Major.Minor)\n * h. Warp size\n */\n\n#include <stdio.h>\n#include <stdlib.h>\n#include <cuda_runtime.h>\n\n#define CHECK_CUDA_ERROR(val) check((val), #val, __FILE__, __LINE__)\nvoid check(cudaError_t result, char const *const func, const char *const file, int const line) {\n    if (result != cudaSuccess) {\n        fprintf(stderr, \"CUDA error at %s:%d code=%d(%s) \\\"%s\\\" \\n\",\n                file, line, (unsigned int)result, \n                cudaGetErrorString(result), func);\n        exit(EXIT_FAILURE);\n    }\n}\n\nint main(int argc, char **argv) {\n    printf(\"====================================================================\\n\");\n    printf(\" CUDA Tutorial - Task 1: CUDA Device Query & Capabilities\\n\");\n    printf(\"====================================================================\\n\\n\");\n\n    int deviceCount = 0;\n    cudaError_t error_id = cudaGetDeviceCount(&deviceCount);\n\n    if (error_id != cudaSuccess) {\n        printf(\"cudaGetDeviceCount returned %d -> %s\\n\", \n               (int)error_id, cudaGetErrorString(error_id));\n        printf(\"Result: NO CUDA-capable devices detected or NVIDIA drivers not installed.\\n\");\n        return 1;\n    }\n\n    if (deviceCount == 0) {\n        printf(\"There are no available CUDA-capable devices detected on this host.\\n\");\n        return 0;\n    }\n\n    printf(\"Detected %d CUDA-capable device(s) on this system.\\n\\n\", deviceCount);\n\n    for (int dev = 0; dev < deviceCount; ++dev) {\n        cudaSetDevice(dev);\n        cudaDeviceProp deviceProp;\n        CHECK_CUDA_ERROR(cudaGetDeviceProperties(&deviceProp, dev));\n\n        printf(\"--------------------------------------------------------------------\\n\");\n        printf(\" Device %d: \\\"%s\\\"\\n\", dev, deviceProp.name);\n        printf(\"--------------------------------------------------------------------\\n\");\n\n        // g. Compute Capability\n        printf(\"  g. Compute Capability:                     %d.%d\\n\", \n               deviceProp.major, deviceProp.minor);\n\n        // b. Total Global Memory\n        printf(\"  b. Total Global Memory:                    %.2f GB (%llu bytes)\\n\",\n               (float)deviceProp.totalGlobalMem / (1024.0f * 1024.0f * 1024.0f),\n               (unsigned long long)deviceProp.totalGlobalMem);\n\n        // c. Shared Memory per Block\n        printf(\"  c. Shared Memory per Block:                %.2f KB (%zu bytes)\\n\",\n               (float)deviceProp.sharedMemPerBlock / 1024.0f,\n               deviceProp.sharedMemPerBlock);\n\n        // d. Number of Multiprocessors (SMs)\n        printf(\"  d. Multiprocessor (SM) Count:              %d\\n\",\n               deviceProp.multiProcessorCount);\n\n        // e. Maximum Threads per Block\n        printf(\"  e. Maximum Threads per Block:              %d\\n\",\n               deviceProp.maxThreadsPerBlock);\n\n        // Maximum Block Dimensions\n        printf(\"     Maximum Block Dimensions (x, y, z):     [%d, %d, %d]\\n\",\n               deviceProp.maxThreadsDim[0], deviceProp.maxThreadsDim[1], deviceProp.maxThreadsDim[2]);\n\n        // f. Maximum Grid Dimensions\n        printf(\"  f. Maximum Grid Dimensions (x, y, z):      [%d, %d, %d]\\n\",\n               deviceProp.maxGridSize[0], deviceProp.maxGridSize[1], deviceProp.maxGridSize[2]);\n\n        // h. Warp Size\n        printf(\"  h. Warp Size:                              %d threads\\n\",\n               deviceProp.warpSize);\n\n        // Additional valuable architectural details (CUDA 12/13+ compliant)\n        printf(\"  - Total Constant Memory:                   %zu bytes\\n\", deviceProp.totalConstMem);\n        printf(\"  - L2 Cache Size:                           %d KB\\n\", deviceProp.l2CacheSize / 1024);\n        printf(\"  - Memory Bus Width:                        %d-bit\\n\", deviceProp.memoryBusWidth);\n        printf(\"  - Maximum Memory Pitch:                    %zu bytes\\n\", deviceProp.memPitch);\n        printf(\"  - Concurrent Copy and Execution:           %s (Async Engines: %d)\\n\", \n               deviceProp.asyncEngineCount > 0 ? \"Yes\" : \"No\", deviceProp.asyncEngineCount);\n        printf(\"  - Integrated / Discrete GPU:               %s\\n\", deviceProp.integrated ? \"Integrated\" : \"Discrete\");\n        printf(\"--------------------------------------------------------------------\\n\\n\");\n    }\n\n    printf(\"Device query completed successfully.\\n\");\n    printf(\"====================================================================\\n\");\n    return 0;\n}\n", "src/02_threads_and_blocks.cu": "/**\n * Tutorial-CUDA: Task 2 (Problem 1) - Exploring Threads and Blocks\n * \n * Demonstrates how CUDA organizes computation into Grids, Blocks, and Threads,\n * and how each thread uniquely identifies itself using built-in variables:\n * - threadIdx: Index of thread within its block (x, y, z)\n * - blockIdx:  Index of block within the grid (x, y, z)\n * - blockDim:  Dimensions of each block in threads (x, y, z)\n * - gridDim:   Dimensions of the grid in blocks (x, y, z)\n */\n\n#include <stdio.h>\n#include <stdlib.h>\n#include <cuda_runtime.h>\n\n#define CHECK_CUDA_ERROR(val) check((val), #val, __FILE__, __LINE__)\nvoid check(cudaError_t result, char const *const func, const char *const file, int const line) {\n    if (result != cudaSuccess) {\n        fprintf(stderr, \"CUDA error at %s:%d code=%d(%s) \\\"%s\\\" \\n\",\n                file, line, (unsigned int)result, \n                cudaGetErrorString(result), func);\n        exit(EXIT_FAILURE);\n    }\n}\n\n// -------------------------------------------------------------\n// Kernel 1: 1D Thread and Block Organization\n// -------------------------------------------------------------\n__global__ void print1DHierarchyKernel(int* d_output) {\n    // Unique global thread ID in 1D\n    int global_id = blockIdx.x * blockDim.x + threadIdx.x;\n    \n    // Store in output array for host verification\n    d_output[global_id] = global_id;\n\n    // Each thread prints its exact coordinates in the hierarchy\n    printf(\"  [1D Kernel] Global ID: %2d | Block ID: %2d (out of %2d) | Thread ID in Block: %2d (out of %2d)\\n\",\n           global_id, blockIdx.x, gridDim.x, threadIdx.x, blockDim.x);\n}\n\n// -------------------------------------------------------------\n// Kernel 2: 2D Thread and Block Organization\n// -------------------------------------------------------------\n__global__ void print2DHierarchyKernel(int width, int height) {\n    // Global 2D coordinates\n    int col = blockIdx.x * blockDim.x + threadIdx.x;\n    int row = blockIdx.y * blockDim.y + threadIdx.y;\n\n    // Linear global ID\n    int global_id = row * width + col;\n\n    // Calculate Warp ID within the block (32 threads per warp)\n    int thread_id_in_block = threadIdx.y * blockDim.x + threadIdx.x;\n    int warp_id = thread_id_in_block / 32;\n\n    if (row < height && col < width) {\n        printf(\"  [2D Kernel] Pos (Row=%d, Col=%d) -> Global Index: %2d | Block: (%d,%d) | Thread in Block: (%d,%d) | Local Rank: %d | Warp: %d\\n\",\n               row, col, global_id, blockIdx.x, blockIdx.y, threadIdx.x, threadIdx.y, thread_id_in_block, warp_id);\n    }\n}\n\nint main(int argc, char** argv) {\n    printf(\"====================================================================\\n\");\n    printf(\" CUDA Tutorial - Task 2 (Problem 1): Threads and Blocks Exploration\\n\");\n    printf(\"====================================================================\\n\\n\");\n\n    // -------------------------------------------------------------\n    // Part A: 1D Thread Hierarchy Demonstration\n    // -------------------------------------------------------------\n    int num_blocks_1d = 3;\n    int threads_per_block_1d = 4;\n    int total_threads_1d = num_blocks_1d * threads_per_block_1d; // 12 threads total\n\n    printf(\"--- Part A: 1D Grid of 1D Blocks ---\\n\");\n    printf(\"Configuration: Grid Dimension = %d blocks, Block Dimension = %d threads\\n\", \n           num_blocks_1d, threads_per_block_1d);\n    printf(\"Total threads executing: %d\\n\\n\", total_threads_1d);\n\n    int* d_output_1d = NULL;\n    int* h_output_1d = (int*)malloc(total_threads_1d * sizeof(int));\n    CHECK_CUDA_ERROR(cudaMalloc((void**)&d_output_1d, total_threads_1d * sizeof(int)));\n\n    // Launch 1D kernel\n    print1DHierarchyKernel<<<num_blocks_1d, threads_per_block_1d>>>(d_output_1d);\n    CHECK_CUDA_ERROR(cudaGetLastError());\n    CHECK_CUDA_ERROR(cudaDeviceSynchronize());\n\n    // Copy back output\n    CHECK_CUDA_ERROR(cudaMemcpy(h_output_1d, d_output_1d, total_threads_1d * sizeof(int), cudaMemcpyDeviceToHost));\n    \n    // Verify on host\n    bool correct_1d = true;\n    for (int i = 0; i < total_threads_1d; i++) {\n        if (h_output_1d[i] != i) correct_1d = false;\n    }\n    printf(\"\\nHost Verification (1D): %s\\n\\n\", correct_1d ? \"SUCCESS (All IDs match linear sequence)\" : \"FAILED\");\n\n    CHECK_CUDA_ERROR(cudaFree(d_output_1d));\n    free(h_output_1d);\n\n    // -------------------------------------------------------------\n    // Part B: 2D Grid of 2D Blocks Hierarchy\n    // -------------------------------------------------------------\n    int matrix_w = 4;\n    int matrix_h = 4;\n\n    dim3 block2D(2, 2); // 4 threads per block (2x2)\n    dim3 grid2D((matrix_w + block2D.x - 1) / block2D.x, \n                (matrix_h + block2D.y - 1) / block2D.y); // 2x2 = 4 blocks\n\n    printf(\"--- Part B: 2D Grid of 2D Blocks ---\\n\");\n    printf(\"Matrix Size: %d x %d (Total %d elements)\\n\", matrix_w, matrix_h, matrix_w * matrix_h);\n    printf(\"Block Dimensions: dim3 block(%d, %d)\\n\", block2D.x, block2D.y);\n    printf(\"Grid Dimensions:  dim3 grid(%d, %d)\\n\", grid2D.x, grid2D.y);\n    printf(\"Total blocks: %d | Total threads: %d\\n\\n\", \n           grid2D.x * grid2D.y, grid2D.x * grid2D.y * block2D.x * block2D.y);\n\n    print2DHierarchyKernel<<<grid2D, block2D>>>(matrix_w, matrix_h);\n    CHECK_CUDA_ERROR(cudaGetLastError());\n    CHECK_CUDA_ERROR(cudaDeviceSynchronize());\n\n    printf(\"\\nExecution completed successfully.\\n\");\n    printf(\"====================================================================\\n\");\n    return 0;\n}\n", "src/03_2d_global_indexing.cu": "/**\n * Tutorial-CUDA: Task 3 (Problem 2a) - 2D Grid-Block Thread Mapping\n * \n * Maps each thread in a 2D grid-block configuration to one element of a 2D matrix\n * and prints its global position (row, column) along with its block and thread indices.\n *\n * Mathematical formulation (Emory CS 355 standard):\n * - Column Index: col = blockIdx.x * blockDim.x + threadIdx.x\n * - Row Index:    row = blockIdx.y * blockDim.y + threadIdx.y\n * - Linear Index: linear_idx = row * width + col\n */\n\n#include <stdio.h>\n#include <stdlib.h>\n#include <cuda_runtime.h>\n\n#define CHECK_CUDA_ERROR(val) check((val), #val, __FILE__, __LINE__)\nvoid check(cudaError_t result, char const *const func, const char *const file, int const line) {\n    if (result != cudaSuccess) {\n        fprintf(stderr, \"CUDA error at %s:%d code=%d(%s) \\\"%s\\\" \\n\",\n                file, line, (unsigned int)result, \n                cudaGetErrorString(result), func);\n        exit(EXIT_FAILURE);\n    }\n}\n\n// -------------------------------------------------------------\n// 2D Matrix Index Mapping Kernel\n// -------------------------------------------------------------\n__global__ void map2DGridBlockKernel(int* d_matrix, int width, int height) {\n    // Compute 2D coordinate for current thread\n    int col = blockIdx.x * blockDim.x + threadIdx.x;\n    int row = blockIdx.y * blockDim.y + threadIdx.y;\n\n    // Boundary check to guard against out-of-bounds access\n    if (row < height && col < width) {\n        int linear_idx = row * width + col;\n        \n        // Store calculated linear index into matrix for verification\n        d_matrix[linear_idx] = linear_idx;\n\n        printf(\"  Thread: (%d, %d) in Block: (%d, %d)  ==>  Matrix Element: [Row %d, Col %d]  |  Linear 1D Index: %2d\\n\",\n               threadIdx.x, threadIdx.y, \n               blockIdx.x, blockIdx.y, \n               row, col, linear_idx);\n    }\n}\n\nint main(int argc, char** argv) {\n    printf(\"====================================================================\\n\");\n    printf(\" CUDA Tutorial - Task 3 (Problem 2a): 2D Grid-Block Matrix Mapping\\n\");\n    printf(\" Reference: Emory CS 355 (2D Grids and Thread Hierarchies)\\n\");\n    printf(\"====================================================================\\n\\n\");\n\n    // Matrix dimensions (6 rows x 8 columns = 48 elements)\n    int width = 8;\n    int height = 6;\n    int total_elements = width * height;\n    size_t bytes = total_elements * sizeof(int);\n\n    // 2D Block dimension: 4 threads in x, 2 threads in y = 8 threads per block\n    dim3 block(4, 2);\n\n    // 2D Grid dimension: calculate blocks needed in x and y (ceiling division)\n    dim3 grid((width + block.x - 1) / block.x, \n              (height + block.y - 1) / block.y);\n\n    printf(\"Matrix Configuration: %d rows x %d columns (%d total elements)\\n\", \n           height, width, total_elements);\n    printf(\"Block Dimension:      dim3 block(%d, %d)  --> %d threads per block\\n\", \n           block.x, block.y, block.x * block.y);\n    printf(\"Grid Dimension:       dim3 grid(%d, %d)   --> %d total blocks\\n\", \n           grid.x, grid.y, grid.x * grid.y);\n    printf(\"Total GPU Threads:    %d threads allocated\\n\\n\", \n           grid.x * grid.y * block.x * block.y);\n\n    int* h_matrix = (int*)malloc(bytes);\n    int* d_matrix = NULL;\n    CHECK_CUDA_ERROR(cudaMalloc((void**)&d_matrix, bytes));\n\n    printf(\"Executing 2D Mapping Kernel and printing thread-to-matrix assignments:\\n\");\n    printf(\"--------------------------------------------------------------------------------\\n\");\n    map2DGridBlockKernel<<<grid, block>>>(d_matrix, width, height);\n    CHECK_CUDA_ERROR(cudaGetLastError());\n    CHECK_CUDA_ERROR(cudaDeviceSynchronize());\n    printf(\"--------------------------------------------------------------------------------\\n\\n\");\n\n    // Copy result back to host and verify\n    CHECK_CUDA_ERROR(cudaMemcpy(h_matrix, d_matrix, bytes, cudaMemcpyDeviceToHost));\n\n    bool verified = true;\n    for (int r = 0; r < height; r++) {\n        for (int c = 0; c < width; c++) {\n            int expected_idx = r * width + c;\n            if (h_matrix[expected_idx] != expected_idx) {\n                verified = false;\n            }\n        }\n    }\n\n    printf(\"Host Verification of 2D Matrix Mappings:\\n\");\n    printf(\"  Verification Status: %s\\n\", verified ? \"PASSED (100% of matrix elements uniquely mapped)\" : \"FAILED\");\n    \n    // Print 2D representation of the mapped matrix\n    printf(\"\\nVisual 2D Matrix Array on Host:\\n\");\n    for (int r = 0; r < height; r++) {\n        printf(\"  Row %d: [ \", r);\n        for (int c = 0; c < width; c++) {\n            printf(\"%2d \", h_matrix[r * width + c]);\n        }\n        printf(\"]\\n\");\n    }\n\n    CHECK_CUDA_ERROR(cudaFree(d_matrix));\n    free(h_matrix);\n\n    printf(\"\\nExecution completed successfully.\\n\");\n    printf(\"====================================================================\\n\");\n    return 0;\n}\n", "src/04_image_brightness.cu": "/**\n * Tutorial-CUDA: Task 4 (Problem 2b) - 2D Image Brightness Adjustment\n * \n * Each thread processes one pixel of a 2D image using a 2D grid-block mapping\n * and increases the brightness/color intensity of each pixel by a fixed value.\n *\n * Problem Specification:\n * - Image Dimension: 512 x 512 = 262,144 pixels\n * - Block Configuration: dim3 block(16, 16)  --> 256 threads per block\n * - Grid Configuration:  dim3 grid(32, 32)   --> 1,024 blocks\n * - Total Threads: 256 x 1024 = 262,144 threads (1:1 mapping per pixel)\n * - Clamping: Pixel values clamped to [0, 255]\n */\n\n#include <stdio.h>\n#include <stdlib.h>\n#include <math.h>\n#include <cuda_runtime.h>\n\n#define WIDTH 512\n#define HEIGHT 512\n#define TOTAL_PIXELS (WIDTH * HEIGHT)\n#define BRIGHTNESS_DELTA 50\n\n#define CHECK_CUDA_ERROR(val) check((val), #val, __FILE__, __LINE__)\nvoid check(cudaError_t result, char const *const func, const char *const file, int const line) {\n    if (result != cudaSuccess) {\n        fprintf(stderr, \"CUDA error at %s:%d code=%d(%s) \\\"%s\\\" \\n\",\n                file, line, (unsigned int)result, \n                cudaGetErrorString(result), func);\n        exit(EXIT_FAILURE);\n    }\n}\n\n// -------------------------------------------------------------\n// CUDA Kernel: 2D Image Brightness Adjustment\n// -------------------------------------------------------------\n__global__ void adjustBrightnessKernel(const unsigned char* __restrict__ d_in, \n                                       unsigned char* __restrict__ d_out, \n                                       int width, int height, int delta) {\n    // 2D coordinate calculation\n    int col = blockIdx.x * blockDim.x + threadIdx.x;\n    int row = blockIdx.y * blockDim.y + threadIdx.y;\n\n    if (row < height && col < width) {\n        int idx = row * width + col;\n        int new_val = (int)d_in[idx] + delta;\n        \n        // Clamping between 0 and 255\n        if (new_val > 255) new_val = 255;\n        if (new_val < 0)   new_val = 0;\n\n        d_out[idx] = (unsigned char)new_val;\n    }\n}\n\n// CPU baseline function for verification and performance comparison\nvoid adjustBrightnessCPU(const unsigned char* h_in, unsigned char* h_out, int width, int height, int delta) {\n    for (int r = 0; r < height; r++) {\n        for (int c = 0; c < width; c++) {\n            int idx = r * width + c;\n            int new_val = (int)h_in[idx] + delta;\n            if (new_val > 255) new_val = 255;\n            if (new_val < 0)   new_val = 0;\n            h_out[idx] = (unsigned char)new_val;\n        }\n    }\n}\n\n// Helper to save image as Portable Graymap (PGM/PPM) format\nvoid savePGM(const char* filename, const unsigned char* image, int width, int height) {\n    FILE* fp = fopen(filename, \"wb\");\n    if (!fp) return;\n    fprintf(fp, \"P5\\n%d %d\\n255\\n\", width, height);\n    fwrite(image, sizeof(unsigned char), width * height, fp);\n    fclose(fp);\n}\n\nint main(int argc, char** argv) {\n    printf(\"====================================================================\\n\");\n    printf(\" CUDA Tutorial - Task 4 (Problem 2b): Image Brightness Adjustment\\n\");\n    printf(\"====================================================================\\n\\n\");\n\n    printf(\"Image Specifications:\\n\");\n    printf(\"  - Dimensions:             %d x %d pixels (Total: %d pixels)\\n\", \n           WIDTH, HEIGHT, TOTAL_PIXELS);\n    printf(\"  - Brightness Adjustment:  %+d intensity units\\n\", BRIGHTNESS_DELTA);\n\n    // Exact block and grid dimensions required by assignment\n    dim3 block(16, 16);\n    dim3 grid(32, 32);\n\n    printf(\"\\nCUDA Launch Geometry:\\n\");\n    printf(\"  - Block Dimensions:       dim3 block(%d, %d) -> %d threads/block\\n\", \n           block.x, block.y, block.x * block.y);\n    printf(\"  - Grid Dimensions:        dim3 grid(%d, %d)  -> %d blocks\\n\", \n           grid.x, grid.y, grid.x * grid.y);\n    printf(\"  - Total Active Threads:   %d threads (1:1 pixel coverage)\\n\\n\",\n           block.x * block.y * grid.x * grid.y);\n\n    size_t img_bytes = TOTAL_PIXELS * sizeof(unsigned char);\n\n    // Host memory allocation\n    unsigned char* h_input = (unsigned char*)malloc(img_bytes);\n    unsigned char* h_output_gpu = (unsigned char*)malloc(img_bytes);\n    unsigned char* h_output_cpu = (unsigned char*)malloc(img_bytes);\n\n    // Generate synthetic gradient and circular image pattern\n    for (int r = 0; r < HEIGHT; r++) {\n        for (int c = 0; c < WIDTH; c++) {\n            int dx = c - WIDTH / 2;\n            int dy = r - HEIGHT / 2;\n            int dist = (int)sqrtf((float)(dx * dx + dy * dy));\n            int val = (r * 128 / HEIGHT) + (dist < 150 ? 80 : 20);\n            if (val > 255) val = 255;\n            h_input[r * WIDTH + c] = (unsigned char)val;\n        }\n    }\n\n    // 1. CPU Sequential Processing\n    printf(\"Executing CPU baseline sequential processing... \");\n    adjustBrightnessCPU(h_input, h_output_cpu, WIDTH, HEIGHT, BRIGHTNESS_DELTA);\n    printf(\"Done.\\n\");\n\n    // 2. CUDA Device Memory Allocation\n    unsigned char* d_input = NULL;\n    unsigned char* d_output = NULL;\n    CHECK_CUDA_ERROR(cudaMalloc((void**)&d_input, img_bytes));\n    CHECK_CUDA_ERROR(cudaMalloc((void**)&d_output, img_bytes));\n\n    // Copy input to GPU\n    CHECK_CUDA_ERROR(cudaMemcpy(d_input, h_input, img_bytes, cudaMemcpyHostToDevice));\n\n    // CUDA Events for high-precision kernel timing\n    cudaEvent_t start_event, stop_event;\n    CHECK_CUDA_ERROR(cudaEventCreate(&start_event));\n    CHECK_CUDA_ERROR(cudaEventCreate(&stop_event));\n\n    // Launch CUDA Kernel\n    CHECK_CUDA_ERROR(cudaEventRecord(start_event, 0));\n    adjustBrightnessKernel<<<grid, block>>>(d_input, d_output, WIDTH, HEIGHT, BRIGHTNESS_DELTA);\n    CHECK_CUDA_ERROR(cudaGetLastError());\n    CHECK_CUDA_ERROR(cudaEventRecord(stop_event, 0));\n    CHECK_CUDA_ERROR(cudaEventSynchronize(stop_event));\n\n    float kernel_time_ms = 0.0f;\n    CHECK_CUDA_ERROR(cudaEventElapsedTime(&kernel_time_ms, start_event, stop_event));\n\n    // Copy result back to host\n    CHECK_CUDA_ERROR(cudaMemcpy(h_output_gpu, d_output, img_bytes, cudaMemcpyDeviceToHost));\n\n    // Verification\n    int error_count = 0;\n    for (int i = 0; i < TOTAL_PIXELS; i++) {\n        if (h_output_gpu[i] != h_output_cpu[i]) {\n            error_count++;\n        }\n    }\n\n    printf(\"\\nExecution Results & Verification:\\n\");\n    printf(\"--------------------------------------------------------------------\\n\");\n    printf(\"  - GPU Kernel Execution Time:   %.4f ms (%.6f seconds)\\n\", \n           kernel_time_ms, kernel_time_ms / 1000.0f);\n    printf(\"  - Total Pixel Processing Rate: %.2f MegaPixels/sec\\n\", \n           ((float)TOTAL_PIXELS / 1e6f) / (kernel_time_ms / 1000.0f));\n    printf(\"  - Verification Status:         %s (Errors: %d / %d)\\n\", \n           (error_count == 0) ? \"PASSED (100% Bitwise Match with CPU)\" : \"FAILED\", \n           error_count, TOTAL_PIXELS);\n    printf(\"  - Sample Pixel Comparison [Pos (256, 256)]:\\n\");\n    printf(\"      Input: %3d  -->  CPU Output: %3d  |  GPU Output: %3d\\n\",\n           h_input[256 * WIDTH + 256], \n           h_output_cpu[256 * WIDTH + 256], \n           h_output_gpu[256 * WIDTH + 256]);\n    printf(\"--------------------------------------------------------------------\\n\\n\");\n\n    // Save sample output image files\n    savePGM(\"input_512x512.pgm\", h_input, WIDTH, HEIGHT);\n    savePGM(\"output_brightened_512x512.pgm\", h_output_gpu, WIDTH, HEIGHT);\n    printf(\"Saved sample test images:\\n\");\n    printf(\"  - input_512x512.pgm\\n\");\n    printf(\"  - output_brightened_512x512.pgm\\n\\n\");\n\n    // Clean up\n    CHECK_CUDA_ERROR(cudaEventDestroy(start_event));\n    CHECK_CUDA_ERROR(cudaEventDestroy(stop_event));\n    CHECK_CUDA_ERROR(cudaFree(d_input));\n    CHECK_CUDA_ERROR(cudaFree(d_output));\n    free(h_input);\n    free(h_output_gpu);\n    free(h_output_cpu);\n\n    printf(\"Execution completed successfully.\\n\");\n    printf(\"====================================================================\\n\");\n    return 0;\n}\n", "src/05_matrix_addition.cu": "/**\n * Tutorial-CUDA: Task 5 (Problem 2i) - 2D Element-Wise Matrix Addition\n * \n * Performs element-wise addition of two matrices A and B (C = A + B)\n * using a 2D grid-block thread mapping where each thread computes\n * one element of the resulting matrix C.\n *\n * Mapping Formula:\n * - col = blockIdx.x * blockDim.x + threadIdx.x\n * - row = blockIdx.y * blockDim.y + threadIdx.y\n * - index = row * width + col\n */\n\n#include <stdio.h>\n#include <stdlib.h>\n#include <math.h>\n#include <cuda_runtime.h>\n\n#define DEFAULT_WIDTH 1024\n#define DEFAULT_HEIGHT 1024\n\n#define CHECK_CUDA_ERROR(val) check((val), #val, __FILE__, __LINE__)\nvoid check(cudaError_t result, char const *const func, const char *const file, int const line) {\n    if (result != cudaSuccess) {\n        fprintf(stderr, \"CUDA error at %s:%d code=%d(%s) \\\"%s\\\" \\n\",\n                file, line, (unsigned int)result, \n                cudaGetErrorString(result), func);\n        exit(EXIT_FAILURE);\n    }\n}\n\n// -------------------------------------------------------------\n// CUDA Kernel: 2D Matrix Addition (C = A + B)\n// -------------------------------------------------------------\n__global__ void matrixAdd2DKernel(const float* __restrict__ A, \n                                  const float* __restrict__ B, \n                                  float* __restrict__ C, \n                                  int width, int height) {\n    int col = blockIdx.x * blockDim.x + threadIdx.x;\n    int row = blockIdx.y * blockDim.y + threadIdx.y;\n\n    if (row < height && col < width) {\n        int idx = row * width + col;\n        C[idx] = A[idx] + B[idx];\n    }\n}\n\n// CPU baseline function for numerical verification\nvoid matrixAddCPU(const float* A, const float* B, float* C, int width, int height) {\n    for (int r = 0; r < height; r++) {\n        for (int c = 0; c < width; c++) {\n            int idx = r * width + c;\n            C[idx] = A[idx] + B[idx];\n        }\n    }\n}\n\nint main(int argc, char** argv) {\n    int width = DEFAULT_WIDTH;\n    int height = DEFAULT_HEIGHT;\n\n    if (argc > 1) width = atoi(argv[1]);\n    if (argc > 2) height = atoi(argv[2]);\n\n    int total_elements = width * height;\n    size_t bytes = total_elements * sizeof(float);\n\n    printf(\"====================================================================\\n\");\n    printf(\" CUDA Tutorial - Task 5 (Problem 2i): 2D Matrix Addition (C = A + B)\\n\");\n    printf(\"====================================================================\\n\\n\");\n\n    printf(\"Matrix Configuration:\\n\");\n    printf(\"  - Dimensions:             %d rows x %d columns\\n\", height, width);\n    printf(\"  - Total Elements:         %d floating-point elements\\n\", total_elements);\n    printf(\"  - Memory per Matrix:      %.2f MB (Total 3 matrices = %.2f MB)\\n\", \n           (float)bytes / (1024.0f * 1024.0f), 3.0f * (float)bytes / (1024.0f * 1024.0f));\n\n    // 2D Block and Grid Launch Configuration\n    dim3 block(16, 16); // 256 threads per block\n    dim3 grid((width + block.x - 1) / block.x, \n              (height + block.y - 1) / block.y);\n\n    printf(\"\\nCUDA Launch Geometry:\\n\");\n    printf(\"  - Block Dimension:        dim3 block(%d, %d) -> %d threads/block\\n\", \n           block.x, block.y, block.x * block.y);\n    printf(\"  - Grid Dimension:         dim3 grid(%d, %d)  -> %d blocks\\n\", \n           grid.x, grid.y, grid.x * grid.y);\n    printf(\"  - Total Threads Launched: %d threads\\n\\n\", \n           grid.x * grid.y * block.x * block.y);\n\n    // Host memory allocation\n    float* h_A = (float*)malloc(bytes);\n    float* h_B = (float*)malloc(bytes);\n    float* h_C_gpu = (float*)malloc(bytes);\n    float* h_C_cpu = (float*)malloc(bytes);\n\n    // Initialize matrices with test values\n    for (int i = 0; i < total_elements; i++) {\n        h_A[i] = (float)(i % 100) * 0.5f;\n        h_B[i] = (float)((total_elements - i) % 100) * 0.25f;\n    }\n\n    // 1. CPU Sequential Baseline Run\n    printf(\"Executing CPU baseline sequential matrix addition... \");\n    matrixAddCPU(h_A, h_B, h_C_cpu, width, height);\n    printf(\"Done.\\n\");\n\n    // 2. Device Memory Allocation\n    float *d_A = NULL, *d_B = NULL, *d_C = NULL;\n    CHECK_CUDA_ERROR(cudaMalloc((void**)&d_A, bytes));\n    CHECK_CUDA_ERROR(cudaMalloc((void**)&d_B, bytes));\n    CHECK_CUDA_ERROR(cudaMalloc((void**)&d_C, bytes));\n\n    // Copy matrices from Host to Device\n    CHECK_CUDA_ERROR(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));\n    CHECK_CUDA_ERROR(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice));\n\n    // Timing with CUDA events\n    cudaEvent_t start_event, stop_event;\n    CHECK_CUDA_ERROR(cudaEventCreate(&start_event));\n    CHECK_CUDA_ERROR(cudaEventCreate(&stop_event));\n\n    // Launch CUDA Kernel\n    CHECK_CUDA_ERROR(cudaEventRecord(start_event, 0));\n    matrixAdd2DKernel<<<grid, block>>>(d_A, d_B, d_C, width, height);\n    CHECK_CUDA_ERROR(cudaGetLastError());\n    CHECK_CUDA_ERROR(cudaEventRecord(stop_event, 0));\n    CHECK_CUDA_ERROR(cudaEventSynchronize(stop_event));\n\n    float kernel_time_ms = 0.0f;\n    CHECK_CUDA_ERROR(cudaEventElapsedTime(&kernel_time_ms, start_event, stop_event));\n\n    // Copy result back from Device to Host\n    CHECK_CUDA_ERROR(cudaMemcpy(h_C_gpu, d_C, bytes, cudaMemcpyDeviceToHost));\n\n    // Numerical Verification\n    float max_error = 0.0f;\n    int error_count = 0;\n    for (int i = 0; i < total_elements; i++) {\n        float diff = fabsf(h_C_gpu[i] - h_C_cpu[i]);\n        if (diff > max_error) max_error = diff;\n        if (diff > 1e-4f) error_count++;\n    }\n\n    // Effective Bandwidth Calculation: 2 reads (A, B) + 1 write (C)\n    double effective_bytes = 3.0 * (double)bytes;\n    double effective_bandwidth_GBs = (effective_bytes / (kernel_time_ms * 1e-3)) / 1e9;\n\n    printf(\"\\nExecution Results & Verification:\\n\");\n    printf(\"--------------------------------------------------------------------\\n\");\n    printf(\"  - GPU Kernel Execution Time:   %.4f ms (%.6f s)\\n\", \n           kernel_time_ms, kernel_time_ms / 1000.0f);\n    printf(\"  - Effective Memory Bandwidth:  %.2f GB/s\\n\", effective_bandwidth_GBs);\n    printf(\"  - Maximum Numerical Error:     %e\\n\", max_error);\n    printf(\"  - Verification Status:         %s (Errors: %d / %d)\\n\", \n           (error_count == 0) ? \"PASSED (100% Numerical Match with CPU)\" : \"FAILED\", \n           error_count, total_elements);\n    printf(\"  - Sample Values [Index (row 0, col 0)]:\\n\");\n    printf(\"      A = %.2f, B = %.2f  -->  C_CPU = %.2f, C_GPU = %.2f\\n\",\n           h_A[0], h_B[0], h_C_cpu[0], h_C_gpu[0]);\n    printf(\"--------------------------------------------------------------------\\n\\n\");\n\n    // Clean up\n    CHECK_CUDA_ERROR(cudaEventDestroy(start_event));\n    CHECK_CUDA_ERROR(cudaEventDestroy(stop_event));\n    CHECK_CUDA_ERROR(cudaFree(d_A));\n    CHECK_CUDA_ERROR(cudaFree(d_B));\n    CHECK_CUDA_ERROR(cudaFree(d_C));\n    free(h_A);\n    free(h_B);\n    free(h_C_gpu);\n    free(h_C_cpu);\n\n    printf(\"Execution completed successfully.\\n\");\n    printf(\"====================================================================\\n\");\n    return 0;\n}\n"}

for rel_path, code in SOURCES.items():
    with open(rel_path, "w", encoding="utf-8") as f:
        f.write(code)
    print(f"✅ Prepared: {rel_path} ({len(code)} bytes)")

print(f"\nWorkspace ready at: {os.getcwd()}")

# Define helper to compile, execute, verify, and save
def compile_and_run(task_num, source_path, binary_path, output_path, success_marker=None):
    print("=" * 70)
    print(f" [TASK {task_num}] Compiling: {source_path} -> {binary_path}")
    print("=" * 70)
    
    # 1. Compile with nvcc -O3
    compile_cmd = ["nvcc", "-O3", source_path, "-o", binary_path]
    comp_res = subprocess.run(compile_cmd, capture_output=True, text=True)
    
    if comp_res.returncode != 0:
        print(f"❌ COMPILATION ERROR in {source_path} (Return Code: {comp_res.returncode}):")
        print(comp_res.stderr)
        raise RuntimeError(f"Compilation failed for {source_path}")
    
    print(f"✅ Compilation succeeded! Binary generated at: {binary_path}")
    print("-" * 70)
    print(f" [TASK {task_num}] Executing on NVIDIA GPU: {binary_path}")
    print("-" * 70)
    
    # 2. Execute binary
    run_res = subprocess.run([f"./{binary_path}"], capture_output=True, text=True)
    combined_output = run_res.stdout + run_res.stderr
    print(combined_output)
    
    if run_res.returncode != 0:
        print(f"❌ RUNTIME ERROR in {binary_path} (Return Code: {run_res.returncode})")
        raise RuntimeError(f"Runtime execution failed for {binary_path}")
    
    # 3. Check verification marker if given
    if success_marker:
        if success_marker in combined_output:
            print(f"✅ Verification check PASSED ('{success_marker}' verified in output)")
        else:
            print(f"⚠️ Warning: Expected marker '{success_marker}' was not found.")
            
    # 4. Save genuine output log
    with open(output_path, "w", encoding="utf-8") as out_f:
        out_f.write(combined_output)
    
    out_size = os.path.getsize(output_path)
    print(f"📁 Genuine output saved to: {output_path} ({out_size} bytes)\n")
    return combined_output


## Task 1: Query CUDA Device Properties
Interrogates device properties via `cudaGetDeviceProperties`.

In [ ]:
compile_and_run(
    task_num=1,
    source_path='src/01_device_query.cu',
    binary_path='bin/01_device_query',
    output_path='results/01_device_query_output.txt',
    success_marker='Device query completed successfully.'
)

## Task 2 (Problem 1): Threads, Blocks, and Built-In Variables
Demonstrates 1D and 2D grid/block hierarchies and built-in variables.

In [ ]:
compile_and_run(
    task_num=2,
    source_path='src/02_threads_and_blocks.cu',
    binary_path='bin/02_threads_and_blocks',
    output_path='results/02_threads_and_blocks_output.txt',
    success_marker='Host Verification (1D): SUCCESS'
)

## Task 3 (Problem 2a): 2D Grid-Block Thread Mapping
Calculates row and column from 2D indexing based on Emory CS 355 curriculum.

In [ ]:
compile_and_run(
    task_num=3,
    source_path='src/03_2d_global_indexing.cu',
    binary_path='bin/03_2d_global_indexing',
    output_path='results/03_2d_global_indexing_output.txt',
    success_marker='Verification Status: PASSED'
)

## Task 4 (Problem 2b): 2D Image Brightness Adjustment (512x512)
Adjusts image brightness with `dim3 block(16, 16)` and `dim3 grid(32, 32)`.

In [ ]:
compile_and_run(
    task_num=4,
    source_path='src/04_image_brightness.cu',
    binary_path='bin/04_image_brightness',
    output_path='results/04_image_brightness_output.txt',
    success_marker='Verification Status:         PASSED'
)

## Task 5 (Problem 2i): 2D Element-Wise Matrix Addition
Computes $C = A + B$ using 2D grid and block indexing.

In [ ]:
compile_and_run(
    task_num=5,
    source_path='src/05_matrix_addition.cu',
    binary_path='bin/05_matrix_addition',
    output_path='results/05_matrix_addition_output.txt',
    success_marker='Verification Status:         PASSED'
)

## Step 3: Verify All Output Files and Download Results ZIP
Checks that all 5 results files are non-empty and downloads `cuda_gpu_results.zip`.

In [ ]:
import os
import zipfile
from google.colab import files

expected_files = [
    'results/01_device_query_output.txt',
    'results/02_threads_and_blocks_output.txt',
    'results/03_2d_global_indexing_output.txt',
    'results/04_image_brightness_output.txt',
    'results/05_matrix_addition_output.txt'
]

print('='*70)
print(' VERIFICATION SUMMARY: Output Files in results/')
print('='*70)
all_valid = True
for ef in expected_files:
    if os.path.exists(ef):
        sz = os.path.getsize(ef)
        status = '✅ NON-EMPTY' if sz > 0 else '❌ EMPTY'
        print(f'{ef:40s} : {status} ({sz:,} bytes)')
        if sz == 0:
            all_valid = False
    else:
        print(f'{ef:40s} : ❌ MISSING')
        all_valid = False

if all_valid:
    print('\n🎉 All 5 CUDA programs executed and produced non-empty genuine output logs!')
    zip_path = 'cuda_gpu_results.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for ef in expected_files:
            zipf.write(ef)
    print(f'📦 Created archive: {zip_path} ({os.path.getsize(zip_path)} bytes)')
    print('⬇️ Triggering automatic download to your computer...')
    files.download(zip_path)
else:
    print('\n⚠️ Some output files are empty or missing. Please check cell outputs above.')
